In [ ]:
import json
import gzip
import pandas as pd
import requests

dataset_category = "Electronics"

### Working with downloaded data

In [ ]:
with gzip.open('../../data/meta_Electronics.jsonl.gz') as f:
  first_line = json.loads(f.readline())

In [ ]:
first_line

### Filter out only recent items (appeared in stock in 2022 or 2023)

In [ ]:
def filter_data_by_date(data: dict) -> bool:
  filter = False
  if int(data["details"]["Date First Available"][-4:]) < 2022:
    filter = True

  return filter

In [ ]:
with gzip.open("../../data/meta_Electronics.jsonl.gz", 'rt') as fp:
  with open("../../data/meta_Electronics_2022_2023.jsonl", 'a', encoding='utf-8') as fp_out:
    with open("../../data/meta_Electronics_no_date.jsonl", 'a', encoding='utf-8') as fp_out_no_date:
      i = 0
      for line in fp:
        curr_electronic = json.loads(line.strip())
        try:
          filter = filter_data_by_date(curr_electronic)
          if not filter:
            json.dump(curr_electronic, fp_out)
            fp_out.write('\n')
            fp_out.flush()
        except:
          json.dump(curr_electronic, fp_out_no_date)
          fp_out_no_date.write('\n')
          fp_out_no_date.flush()
        i += 1
        if i % 10000 == 0:
          print(f"Processed {i} lines")



### Split the items into two categories: "has main category", "does not have main category"

In [ ]:
def filter_category(data:dict) -> bool:
  filter = False
  if data["main_category"] == None:
    filter = True
  return filter




In [ ]:
with open("../../data/meta_Electronics_2022_2023.jsonl", 'r') as fp:
  with open("../../data/meta_Electronics_2022_2023_with_category.jsonl", 'a', encoding='utf-8') as fp_out:
    with open("../../data/meta_Electronics_2022_2023_no_category.jsonl", 'a', encoding='utf-8') as fp_out_no_category:
      i = 0
      for line in fp:
        curr_electronic = json.loads(line.strip())
        if not filter_category(curr_electronic):
          json.dump(curr_electronic, fp_out)
          fp_out.write('\n')
          fp_out.flush()
        else:
          json.dump(curr_electronic, fp_out_no_category)
          fp_out_no_category.write('\n')
          fp_out_no_category.flush()
        i += 1  
        if i % 10000 == 0:
          print(f"Processed {i} lines")





### Explore distribution by category


In [ ]:
df = pd.read_json("../../data/meta_Electronics_2022_2023_with_category.jsonl", lines=True)

In [ ]:
df.head()

In [ ]:
df["main_category"].value_counts().plot(kind="bar")

### Filter out items that have at least 100 ratings

In [ ]:
df_ratings_100 = df[df["rating_number"] >= 100]
len(df_ratings_100)


In [ ]:
df_ratings_100["main_category"].value_counts().plot( # pyright: ignore[reportAttributeAccessIssue]
    kind="bar"
) 

### Explore Distribution of ratings

In [ ]:
average_ratings_column = df_ratings_100["average_rating"]
if not isinstance(average_ratings_column, pd.Series):
    raise TypeError("expected a Series")
average_ratings_column.plot(kind="hist", bins=50, range=(0, 5))

### Sample 1000

In [ ]:
df_sample_1000 = df_ratings_100.sample(n=1000, random_state=42)

In [ ]:
len(df_sample_1000)

In [ ]:
df_sample_1000["average_rating"].plot(kind="hist", bins=50, range=(0, 5)) # pyright: ignore[reportAttributeAccessIssue]

In [ ]:
df_sample_1000["price"].plot(kind="hist", bins=100, range=(0, 500)) # pyright: ignore[reportAttributeAccessIssue]

In [ ]:
df_sample_1000["main_category"].value_counts().plot(kind="bar") # pyright: ignore[reportAttributeAccessIssue]

In [ ]:
df_ratings_100.to_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100.jsonl", orient="records", lines=True)
df_sample_1000.to_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", orient="records", lines=True)

### Extract ratings that match sampled data

In [ ]:
df_ratings_100 = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100.jsonl", lines=True)
df_sample_1000 = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [ ]:
df_ratings_100.head()


In [ ]:
df_sample_1000.head()

In [ ]:
with gzip.open("../../data/Electronics.jsonl.gz", "r") as fp_electronics:
    with open(
        "../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl",
        "a",
    ) as fp_out:
        asins_in_sample = set(df_sample_1000["parent_asin"].values)
        i = 0
        for line in fp_electronics:
            curr_electronic = json.loads(line.strip())
            if curr_electronic["parent_asin"] in asins_in_sample:
                json.dump(curr_electronic, fp_out)
                fp_out.write("\n")
                fp_out.flush()
            i += 1
            if i % 10000 == 0:
                print(f"Processed {i} lines")